# Spike Sorting **v2** — developer walkthrough

> **Audience:** developers of the `spyglass.spikesorting.v2` pipeline.
> This is an *internals* walkthrough, not a user tutorial. The goal is to make the
> **table structure** and the **dependency flow** concrete by driving the pipeline
> one DataJoint table at a time — i.e. doing by hand exactly what
> [`run_v2_pipeline`](../src/spyglass/spikesorting/v2/pipeline.py) collapses into a
> single call.
>
> v2 is **pre-release**. Table definitions can still change; treat the source under
> `src/spyglass/spikesorting/v2/` as the authority if anything here drifts.

## The shape of the pipeline

Every compute stage follows the same **three-table pattern** Spyglass uses everywhere:

| role | tier | what it is | example |
|------|------|-----------|---------|
| **Parameters** | `Lookup` | reusable, named knob bundles | `PreprocessingParameters`, `SorterParameters` |
| **Selection** | `Manual` | "compute *this* with *these* params" — one row = one job | `RecordingSelection`, `SortingSelection` |
| **Result** | `Computed` | the materialized output; filled by `.populate()` | `Recording`, `Sorting` |

A stage is run by **inserting a Selection row** and then **calling `.populate()`** on the
Computed table. `populate()` finds Selection rows with no matching Computed row, runs
`make()`, and writes the result. It is idempotent — re-running is a no-op.

## The dependency DAG (what depends on what)

```
   common.Session ─┐
   common.Raw ─────┤
   SortGroupV2 ─────┤                          (Lookup params feed each Selection)
   IntervalList ───┤   PreprocessingParameters ─┐
   common.LabTeam ─┴──▶ RecordingSelection ◀────┘
                              │  .populate()
                              ▼
                          Recording ───────────────┐
                              │                     │   ArtifactDetectionParameters ─┐
                              │                ▶ ArtifactSelection ◀─────────────────┘
                              │                     │  .populate()
                              │                     ▼
                              │                ArtifactDetection ──▶ writes common.IntervalList
                              │                     │                  ("artifact_<uuid>")
                              ▼                     ▼   SorterParameters ─┐
                          (RecordingSource)  (ArtifactSource)            │
                                   └──────▶ SortingSelection ◀───────────┘
                                                  │  .populate()
                                                  ▼
                                              Sorting ──▶ Sorting.Unit (one row / unit)
                                                  │
                                                  ▼   CurationV2.insert_curation()
                                              CurationV2 ──▶ .Unit / .UnitLabel / .MergeGroup
                                                  │              (and registers the merge row)
                                                  ▼
                                       SpikeSortingOutput  (merge master; merge_id = downstream handle)
```

**Selection tables are UUID-keyed and content-addressed.** `RecordingSelection`,
`ArtifactSelection`, and `SortingSelection` derive their primary key (`recording_id` /
`artifact_id` / `sorting_id`) deterministically from the *content* of the selection (a
uuid5 over the resolved upstream keys + params). That is what makes `insert_selection`
a **find-or-insert** and the whole pipeline idempotent: the same inputs always map to the
same UUID, so re-running never duplicates rows. We will see this directly at the end.

We will walk the DAG top-to-bottom and inspect every table as we fill it.

---
## 0. Environment — connect to an **isolated** database (never production)

This notebook drives `.populate()`, which **writes** to the database and the analysis
directory. To keep that safe, we use the same bootstrap the v2 test suite and fixture
generator use: [`bootstrap_v2_test_environment`](../tests/spikesorting/v2/test_env.py).

It:
- refuses any base dir that looks like shared lab storage (`stelmo`, …) or sits outside
  `tests/_data/` or a tempdir,
- refuses any schema prefix that is not `pytests` / `test*`,
- **starts or reuses** the Colima/Docker MySQL test container and pins `database.prefix`,
- repoints `SPYGLASS_BASE_DIR` at our throwaway base dir.

**Prereqs on this machine** (see the `spikesorting-v2-local-test-env` note):
launch Jupyter from the `spyglass_spikesorting_v2` conda env, after
`source ~/spyglass_v2_env.sh` (sets `DOCKER_HOST` for Colima + puts NEURON on `PATH`),
with Colima running.

> Already have your own `dj_local_conf.json` pointed at a **disposable** database? You can
> skip this cell — but then *you* are responsible for not pointing at production.

In [ ]:
import sys
from pathlib import Path

# Repo root = three levels up from notebooks/ (this file lives in notebooks/).
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "spyglass").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
print("repo root:", REPO_ROOT)

# The bootstrap helper lives next to the v2 tests and must run BEFORE importing spyglass.
V2_TEST_DIR = REPO_ROOT / "tests" / "spikesorting" / "v2"
sys.path.insert(0, str(V2_TEST_DIR))
sys.path.insert(0, str(REPO_ROOT))

from test_env import bootstrap_v2_test_environment  # noqa: E402

# Throwaway base dir under tests/_data/ (allowed by the safety check).
NOTEBOOK_BASE_DIR = REPO_ROOT / "tests" / "_data" / "spikesorting_v2_notebook"
bootstrap_v2_test_environment(NOTEBOOK_BASE_DIR, database_prefix="pytests")

import datajoint as dj  # noqa: E402

print("database.prefix :", dj.config["database.prefix"])  # must be 'pytests'
print("SPYGLASS_BASE_DIR:", NOTEBOOK_BASE_DIR)

In [ ]:
# Quiet, notebook-friendly logging, and import the pieces we will touch.
import warnings

import datajoint as dj

from spyglass.common import Session, Raw, IntervalList, LabTeam
from spyglass.common.common_ephys import Electrode

warnings.filterwarnings("ignore", category=UserWarning)
dj.config["display.limit"] = 12  # cap rows in table reprs

---
## 1. Get the **MEArec** data into Spyglass (`insert_sessions`)

The v2 suite validates against **MEArec** ground-truth simulations converted to
Spyglass-ingestible NWB. `_fixtures/mearec_to_nwb.py` writes an NWB that is structurally
identical to a `trodes_to_nwb` file (one `ndx_franklab_novela` probe, an `Electrode`
table, an `ElectricalSeries` named `"e-series"`), plus a *sidecar* ground-truth `Units`
table so the planted spikes can be used as an accuracy oracle.

We use the committed **smoke** fixture (`mearec_polymer_smoke.nwb`): a 4 s, 128-channel,
4-shank polymer probe with 6 planted units. It is the fastest fixture and the one the
`populated_sorting` conftest fixture exercises end-to-end, so we know this exact path runs.

`copy_and_insert_nwb` (a test helper) copies the NWB into `$SPYGLASS_RAW_DIR` and runs
`insert_sessions(..., reinsert=True)`. Ingestion populates the **upstream common tables**
the whole pipeline sits on: `Session`, `Nwbfile`, `Raw`, `Electrode`, and the
`"raw data valid times"` `IntervalList`.

> The `.nwb` fixtures are git-ignored. If the cell below skips, generate the smoke fixture
> with `python tests/spikesorting/v2/fixtures/generate_mearec.py --smoke`.

In [ ]:
from tests.spikesorting.v2._ingest_helpers import copy_and_insert_nwb

FIXTURE_PATH = V2_TEST_DIR / "fixtures" / "mearec_polymer_smoke.nwb"
assert FIXTURE_PATH.exists(), (
    f"Missing {FIXTURE_PATH.name}. Generate it with:\n"
    "  python tests/spikesorting/v2/fixtures/generate_mearec.py --smoke"
)

# Ingest under a dedicated name so we never clobber rows the test suite owns.
nwb_file_name = copy_and_insert_nwb(FIXTURE_PATH, dest_name="mearec_v2_walkthrough.nwb")
session_key = {"nwb_file_name": nwb_file_name}
print("ingested as:", nwb_file_name)

In [ ]:
# What ingestion created upstream of the v2 pipeline:
print("Session       :", len(Session & session_key))
print("Raw           :", len(Raw & session_key))
print("Electrode     :", len(Electrode & session_key), "contacts")
print("IntervalLists :", (IntervalList & session_key).fetch("interval_list_name"))
Session & session_key

---
## 2. Seed the **Lookup** parameter tables + a `LabTeam`

`Lookup` tables hold reusable, named parameter presets shared across sessions. Each is
keyed by a single name string and carries a validated `params` blob plus a
`params_schema_version`. `initialize_v2_defaults()` installs the default rows for every
stage in one idempotent call so you never hit a "forgot to `insert_default`" FK error.

We also need a `LabTeam` row — `RecordingSelection` carries a `-> LabTeam` foreign key
(v2 makes the owning team part of the recording's identity).

In [ ]:
from spyglass.spikesorting.v2 import initialize_v2_defaults
from spyglass.spikesorting.v2.recording import PreprocessingParameters
from spyglass.spikesorting.v2.artifact import ArtifactDetectionParameters
from spyglass.spikesorting.v2.sorting import SorterParameters

initialize_v2_defaults()  # PreprocessingParameters / ArtifactDetection / Sorter / Motion

TEAM_NAME = "v2_walkthrough_team"
LabTeam.insert1(
    {"team_name": TEAM_NAME, "team_description": "spike sorting v2 walkthrough"},
    skip_duplicates=True,
)

print("preproc presets :", list(PreprocessingParameters.fetch("preproc_params_name")))
print("artifact presets:", list(ArtifactDetectionParameters.fetch("artifact_params_name")))
print("sorter presets  :", sorted(set(SorterParameters.fetch("sorter"))))

A `Lookup` row is just `name -> (params, schema_version)`. Note `params_schema_version`:
it lets the pipeline evolve the params schema without silently reinterpreting old blobs.

In [ ]:
print(PreprocessingParameters.describe())
PreprocessingParameters & {"preproc_params_name": "default_franklab"}

### The whole v2 chain, as DataJoint sees it

`dj.Diagram` renders the live foreign-key graph. Solid lines are FK dependencies; the
dotted box rows are `Part` tables (master/part). If `graphviz` is not installed the cell
falls back to printing the edge list.

In [ ]:
from spyglass.spikesorting.v2.recording import RecordingSelection, Recording, SortGroupV2
from spyglass.spikesorting.v2.artifact import ArtifactSelection, ArtifactDetection
from spyglass.spikesorting.v2.sorting import SortingSelection, Sorting
from spyglass.spikesorting.v2.curation import CurationV2
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput

try:
    diagram = (
        dj.Diagram(SortGroupV2)
        + dj.Diagram(RecordingSelection)
        + dj.Diagram(Recording)
        + dj.Diagram(ArtifactSelection)
        + dj.Diagram(ArtifactDetection)
        + dj.Diagram(SortingSelection)
        + dj.Diagram(Sorting)
        + dj.Diagram(CurationV2)
        + dj.Diagram(SpikeSortingOutput)
    )
    display(diagram)
except Exception as exc:  # graphviz missing, etc.
    print("Diagram render failed (need graphviz?):", exc)
    print("Falling back to the dependency edge list in the markdown above.")

---
## Stage A — `SortGroupV2`: which electrodes get sorted together

`SortGroupV2` is a **Manual** table (you populate it, not `.populate()`). It groups the
session's electrodes into the units that get sorted as one — typically one group per shank
/ tetrode. It is a **master/part** table:

- **master** `SortGroupV2` — `(-> Session, sort_group_id)` + reference config
- **part** `SortGroupV2.SortGroupElectrode` — `(-> master, -> Electrode)`: the membership

`set_group_by_shank` reads the `Electrode` table's `probe_shank` column and makes one group
per shank. v2 hardens v1's silent-overwrite bug: re-running on a session that already has
groups **raises** instead of quietly extending — you must pass `delete_existing_entries=True,
confirm=True` to redo it.

In [ ]:
from spyglass.spikesorting.v2.recording import SortGroupV2

# Idempotent guard: only build groups if this session has none yet.
if not (SortGroupV2 & session_key):
    SortGroupV2.set_group_by_shank(nwb_file_name=nwb_file_name)

masters = SortGroupV2 & session_key
members = SortGroupV2.SortGroupElectrode & session_key
print(f"{len(masters)} sort groups, {len(members)} electrodes total")

# Pick the first shank to carry through the rest of the walkthrough.
SORT_GROUP_ID = int(sorted(masters.fetch("sort_group_id"))[0])
print("using sort_group_id =", SORT_GROUP_ID,
      "with", len(members & {"sort_group_id": SORT_GROUP_ID}), "electrodes")
masters

In [ ]:
# The definition: master PK is (Session, sort_group_id); the part hangs Electrode rows off it.
print("=== SortGroupV2 (master) ===")
print(SortGroupV2.describe())
print("\n=== SortGroupV2.SortGroupElectrode (part) ===")
print(SortGroupV2.SortGroupElectrode.describe())

---
## Stage B — `Recording`: materialize the preprocessed traces

This is the first **Selection → Computed** stage, and the template for every one after it.

**`RecordingSelection`** (`Manual`, UUID-keyed) is one row = "preprocess *this* sort group
of *this* raw session over *this* interval with *these* params, owned by *this* team". Its
PK `recording_id` is a deterministic uuid5 of that bundle, so `insert_selection` is
find-or-insert. Note the secondary attributes are all foreign keys — the selection's whole
job is to *bind* the upstream rows:

```
recording_id : uuid          # = uuid5(resolved upstream keys + params)
---
-> Raw
-> SortGroupV2
-> IntervalList
-> PreprocessingParameters
-> LabTeam
```

**`Recording`** (`Computed`) is filled by `.populate()`. Its `make()` is split three ways
— `make_fetch` (gather DB inputs), `make_compute` (the heavy SpikeInterface work — bandpass
+ common-reference — done *outside* the DB transaction), `make_insert` (atomic write). It
streams the preprocessed traces into a fresh `AnalysisNwbfile` and records a `cache_hash`
of the recipe so the materialized artifact can be validated/rebuilt.

In [ ]:
PREPROC_PARAMS = "default_franklab"

rec_pk = RecordingSelection.insert_selection(
    {
        "nwb_file_name": nwb_file_name,
        "sort_group_id": SORT_GROUP_ID,
        "interval_list_name": "raw data valid times",
        "preproc_params_name": PREPROC_PARAMS,
        "team_name": TEAM_NAME,
    }
)
print("recording_id (deterministic uuid):", rec_pk["recording_id"])
RecordingSelection & rec_pk

In [ ]:
# Fill the Computed table. reserve_jobs=False keeps it simple for an interactive run.
Recording.populate(rec_pk, reserve_jobs=False)

print(Recording.describe())
Recording & rec_pk

In [ ]:
# The Computed row points at an AnalysisNwbfile and records the materialized recording's
# shape + a cache_hash. get_recording() rehydrates the actual SpikeInterface object.
row = (Recording & rec_pk).fetch1()
print("n_channels        :", row["n_channels"])
print("sampling_frequency:", row["sampling_frequency"])
print("duration_s        :", round(float(row["duration_s"]), 3))
print("cache_hash        :", row["cache_hash"][:16], "...")

si_recording = Recording().get_recording(rec_pk)
print("\nSpikeInterface recording:", si_recording)

---
## Stage C — `ArtifactDetection`: mask out artifact times

Same Selection → Computed shape, with a twist: **`ArtifactSelection` is source-polymorphic.**
Its PK is just `artifact_id : uuid`, and the *source* is attached as exactly one `Part` row:

- `ArtifactSelection.RecordingSource` → a single `Recording` (our case), or
- `ArtifactSelection.SharedArtifactGroupSource` → a bundle of recordings that share a pass.

`insert_selection` enforces the exactly-one-source rule and folds the source into the
deterministic `artifact_id`.

**`ArtifactDetection`** (`Computed`) runs amplitude/z-score thresholding and writes the
*artifact-removed valid times* back into `common.IntervalList`, under the name
`"artifact_<artifact_id>"`. That is the v2 contract: downstream sorting consumes an
`IntervalList`, so artifact detection is just "produce a cleaner interval".

We use the `"none"` preset here (detection disabled → pass-through valid times). It is the
fast, guaranteed path the `populated_sorting` fixture uses; swap to `"default"` to actually
threshold.

In [ ]:
ARTIFACT_PARAMS = "none"  # pass-through; use "default" to actually threshold

art_pk = ArtifactSelection.insert_selection(
    {
        "recording_id": rec_pk["recording_id"],
        "artifact_params_name": ARTIFACT_PARAMS,
    }
)
print("artifact_id:", art_pk["artifact_id"])
print("source part (RecordingSource):")
display(ArtifactSelection.RecordingSource & art_pk)
ArtifactSelection & art_pk

In [ ]:
ArtifactDetection.populate(art_pk, reserve_jobs=False)

# The output is a new IntervalList row named after the artifact_id.
art_interval_name = f"artifact_{art_pk['artifact_id']}"
print("wrote IntervalList:", art_interval_name)
IntervalList & session_key & {"interval_list_name": art_interval_name}

---
## Stage D — `Sorting`: run the sorter

**`SortingSelection`** (`Manual`, UUID-keyed) binds a recording source, an optional
artifact source, and a `SorterParameters` row. Like artifact selection it is
source-polymorphic via parts:

- `SortingSelection.RecordingSource` (single recording) **xor**
  `SortingSelection.ConcatenatedRecordingSource` (multi-session concat — not yet wired), and
- `SortingSelection.ArtifactSource` (optional — the interval mask to honor).

`insert_selection` takes flat kwargs (`recording_id`, `sorter`, `sorter_params_name`,
`artifact_id`), resolves them to the right parts, and folds everything into `sorting_id`.

**`Sorting`** (`Computed`) applies any post-motion preprocessing, runs the SpikeInterface
sorter, removes excess spikes, builds a `SortingAnalyzer`, and writes a units NWB. It is a
master/part table — `Sorting.Unit` carries one row per sorted unit:
`(unit_id) -> Electrode, peak_amplitude_uv, n_spikes`.

In [ ]:
SORTER = "mountainsort5"
SORTER_PARAMS = "franklab_tetrode_hippocampus_30kHz_ms5"

sort_pk = SortingSelection.insert_selection(
    {
        "recording_id": rec_pk["recording_id"],
        "sorter": SORTER,
        "sorter_params_name": SORTER_PARAMS,
        "artifact_id": art_pk["artifact_id"],
    }
)
print("sorting_id:", sort_pk["sorting_id"])
print("recording source:", len(SortingSelection.RecordingSource & sort_pk),
      "| artifact source:", len(SortingSelection.ArtifactSource & sort_pk))
SortingSelection & sort_pk

In [ ]:
# This actually runs the sorter — the slowest cell (tens of seconds on the smoke fixture).
Sorting.populate(sort_pk, reserve_jobs=False)

n_units = int((Sorting & sort_pk).fetch1("n_units"))
print("n_units:", n_units)
print(Sorting.describe())
Sorting & sort_pk

In [ ]:
# One row per sorted unit. (Zero units is a legitimate result on a quiet 4 s shank — the
# pipeline handles it; if you see 0 here, try sort_group_id of a busier shank or "default"
# artifact params, or the 60 s fixtures.)
Sorting.Unit & sort_pk

---
## Stage E — `CurationV2`: label / merge, and register the output

`CurationV2` is a **Manual** table, but you don't hand-insert it — you call the
`insert_curation` classmethod, which atomically writes the master, the part rows, **and** the
merge-table registration in one transaction.

It is keyed `(-> Sorting, curation_id)` with a `parent_curation_id` for lineage — a *root*
curation has `parent_curation_id = -1`; later curations point back at an earlier one, so you
get an editable chain without mutating history. Its parts:

- `CurationV2.Unit` — the curated unit set (mirrors `Sorting.Unit`, possibly merged)
- `CurationV2.UnitLabel` — `(-> Unit, curation_label)` validated against the label enum
- `CurationV2.MergeGroup` — records proposed/applied merges as (kept-unit, contributor) pairs

A root curation with no labels and no merges is the minimal "accept the sort as-is" step —
exactly what the orchestrator does by default. Crucially, `insert_curation` also inserts the
`SpikeSortingOutput.CurationV2` row, so the result becomes addressable by a `merge_id`.

In [ ]:
from spyglass.spikesorting.v2.curation import CurationV2

# Idempotent: reuse the existing root curation if we already made one (mirrors
# run_v2_pipeline). insert_curation(reuse_existing=True) is the other option.
existing_root = (CurationV2 & sort_pk & {"parent_curation_id": -1}).fetch(
    "KEY", as_dict=True
)
if existing_root:
    curation_pk = existing_root[0]
else:
    curation_pk = CurationV2.insert_curation(
        sorting_key=sort_pk,
        labels={},               # no manual labels
        parent_curation_id=-1,   # root curation
        description="v2 walkthrough root curation",
    )
print("curation_pk:", curation_pk)
CurationV2 & curation_pk

In [ ]:
print("=== CurationV2 (master) ===")
print(CurationV2.describe())
print("\ncurated units:")
CurationV2.Unit & curation_pk

---
## Stage F — `SpikeSortingOutput`: the merge master (downstream handle)

`SpikeSortingOutput` is the **merge master** that unifies v0 / v1 / v2 spike-sorting
outputs behind a single `merge_id : uuid`. Downstream consumers (decoding, analysis groups)
key off `merge_id` and never need to know which pipeline version produced it. `CurationV2`
plugs in via the `SpikeSortingOutput.CurationV2` part table that `insert_curation` already
populated.

Fetch the `merge_id` from the part, then read the curated result back through the
`CurationV2` accessors (the same ones the merge dispatcher calls).

In [ ]:
merge_id = (SpikeSortingOutput.CurationV2 & curation_pk).fetch1("merge_id")
print("merge_id:", merge_id)
print("source  :", (SpikeSortingOutput & {"merge_id": merge_id}).fetch1("source"))
SpikeSortingOutput & {"merge_id": merge_id}

In [ ]:
# Read the curated sorting back as a DataFrame (one row per unit, with spike times).
units_df = CurationV2.get_sorting(curation_pk, as_dataframe=True)
print("curated units:", len(units_df))
units_df.head()

---
## Recap — the PKs we threaded through the DAG

Every stage handed its primary key to the next as a foreign key. The chain we built:

| stage | table | tier | key we produced |
|-------|-------|------|-----------------|
| ingest | `Session` / `Raw` / `Electrode` / `IntervalList` | Manual/Imported | `nwb_file_name` |
| A | `SortGroupV2` (+ `.SortGroupElectrode`) | Manual | `sort_group_id` |
| B | `RecordingSelection` → `Recording` | Manual → Computed | `recording_id` |
| C | `ArtifactSelection` → `ArtifactDetection` | Manual → Computed | `artifact_id` (+ `IntervalList`) |
| D | `SortingSelection` → `Sorting` (+ `.Unit`) | Manual → Computed | `sorting_id` |
| E | `CurationV2` (+ `.Unit`/`.UnitLabel`/`.MergeGroup`) | Manual | `curation_id` |
| F | `SpikeSortingOutput` | Merge | `merge_id` |

In [ ]:
manifest_by_hand = {
    "recording_id": rec_pk["recording_id"],
    "artifact_id": art_pk["artifact_id"],
    "sorting_id": sort_pk["sorting_id"],
    "curation_id": curation_pk["curation_id"],
    "merge_id": merge_id,
    "n_units": n_units,
}
manifest_by_hand

## The one-call equivalent — `run_v2_pipeline`

[`run_v2_pipeline`](../src/spyglass/spikesorting/v2/pipeline.py) is **exactly the
Stage B–F cells above**, wrapped in a loop over a *preset* (a Pydantic-validated bundle of
the Lookup-row names). It does the `insert_selection` → `populate` chain for recording,
artifact, sorting, then the root-curation insert, and returns the same manifest dict.

It does **not** do the upstream setup — you still need an ingested session,
`initialize_v2_defaults()`, a `LabTeam`, and a `SortGroupV2`. So a single-session sort is
~4 touchpoints (ingest + defaults/team + sort group + this call), with the per-stage
boilerplate collapsed.

### Idempotency, demonstrated

Because the Selection PKs are deterministic uuid5s of their content, calling
`run_v2_pipeline` **twice** returns the byte-identical manifest — no duplicate rows. (Note
the default preset uses `artifact_params_name="default"`, whereas our by-hand chain used
`"none"`; different content → a *different* `artifact_id` and therefore a different
`sorting_id`/`merge_id`. That divergence is the content-addressing working as designed: the
artifact params are part of the sort's identity.)

In [ ]:
from spyglass.spikesorting.v2.pipeline import run_v2_pipeline, list_presets

print("available presets:", list_presets())

manifest_1 = run_v2_pipeline(
    nwb_file_name=nwb_file_name,
    sort_group_id=SORT_GROUP_ID,
    interval_list_name="raw data valid times",
    team_name=TEAM_NAME,
    preset="franklab_tetrode_mountainsort5",
)
manifest_1

In [ ]:
# Run it again — same inputs, same deterministic UUIDs, identical manifest, no new rows.
manifest_2 = run_v2_pipeline(
    nwb_file_name=nwb_file_name,
    sort_group_id=SORT_GROUP_ID,
    interval_list_name="raw data valid times",
    team_name=TEAM_NAME,
    preset="franklab_tetrode_mountainsort5",
)
assert manifest_1 == manifest_2, "run_v2_pipeline should be idempotent!"
print("idempotent ✓  — merge_id:", manifest_2["merge_id"])
print("\nby-hand artifact='none' merge_id :", merge_id)
print("preset  artifact='default' merge_id:", manifest_2["merge_id"])
print("→ different artifact params ⇒ different content-addressed identity, as expected.")

---
## What we did *not* touch (forward-declared in v2)

The schema already carries the multi-session surface, but the compute is gated:

- **`SessionGroup` / `ConcatenatedRecordingSelection` / `ConcatenatedRecording`** — bundle
  several sessions and materialize one concatenated recording. `ConcatenatedRecording.make`
  raises `NotImplementedError` today; `SortingSelection.ConcatenatedRecordingSource` is the
  hook that will feed it into sorting.
- **`MotionCorrectionParameters`** — seeded by `initialize_v2_defaults` so the eventual
  concat path won't FK-fault, but unused until `ConcatenatedRecording` lands.
- **`SharedArtifactGroup`** — cross-recording artifact passes (the second `ArtifactSelection`
  source), for behavioral artifacts shared across a group.
- **metrics + auto-curation / figpack UI** — `CurationV2.metrics_source` and the
  `analyzer_curation` / `figpack` provenance enum anticipate it; the orchestrator only does
  a manual root curation today.

For the per-stage `make()` details, read the tri-split bodies directly:
`recording.py` (`make_fetch`/`make_compute`/`make_insert`), `artifact.py`, `sorting.py`,
and `curation.py::insert_curation`.